# Entrenamiento de hoop-ball.tflite (aro y balon)
Entrena un detector EfficientDet-Lite0 con MediaPipe Model Maker sobre datasets de Roboflow y exporta un modelo int8 de menos de 5 MB.

**Nota de versiones:** Colab ya usa Python 3.13 y `mediapipe-model-maker` solo funciona hasta Python 3.11 (TensorFlow 2.15). Por eso la celda 2 crea un Python 3.11 aparte en `/content/py311` y las celdas que empiezan por `%%py311` se ejecutan con el. La descarga y la unificacion siguen en el Python normal de Colab.

Ejecuta las celdas en orden. Si una falla, para y no sigas con las siguientes.

## 1. Configuracion
Rellena `DATASETS` con el workspace, proyecto y version de Roboflow Universe.

In [ ]:
DATASETS = [
    # "cv" by CV (CC BY 4.0): clases basketball, rim, people. 3 666 imagenes.
    {"workspace": "cv-8scak", "project": "cv-cnfd4", "version": 1},
]
EPOCAS = 50
BATCH = 16
LR = 0.3
MAX_MB = 5
# Rama del repo de donde se baja unificar_etiquetas.py. Cambiala a "main" cuando se fusione.
RAMA_REPO = "feature/modelo-hoop-ball"

assert DATASETS, "Rellena DATASETS con al menos un dataset de Roboflow (workspace, project, version)."

## 2. Entorno
Comprueba la GPU, instala `roboflow`, crea el Python 3.11 con MediaPipe Model Maker y TensorFlow 2.15 (tarda unos minutos), comprueba que ve la GPU y obtiene `unificar_etiquetas.py` (por `git clone` o subiendolo a mano si el repo es privado). Tambien define la magia `%%py311`.

In [ ]:
import subprocess, sys, os, shutil, glob, re
from IPython.core.magic import register_cell_magic

try:
    subprocess.run(["nvidia-smi"], check=True, capture_output=True)
except (FileNotFoundError, subprocess.CalledProcessError):
    raise SystemExit(
        "No hay GPU. En Colab: Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > "
        "Acelerador por hardware: GPU T4. Despues vuelve a ejecutar todo."
    )

VENV = "/content/py311"
PY311 = f"{VENV}/bin/python"
# Versiones fijadas: mediapipe-model-maker 0.2.1.4 necesita TensorFlow < 2.16 (Python <= 3.11)
DEPENDENCIAS_PY311 = [
    "mediapipe-model-maker==0.2.1.4",
    "mediapipe==0.10.14",
    "tensorflow[and-cuda]==2.15.1",
    "tf-models-official==2.15.0",
]
_PROGRESO = re.compile(r"^\s*(\d+)/(\d+) \[")


def correr(cmd, env=None):
    """Ejecuta cmd mostrando su salida en vivo; lanza un error si falla."""
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, cwd=os.getcwd())
    for linea in p.stdout:
        m = _PROGRESO.match(linea)
        # De las barras de progreso de Keras solo se muestra la del final de cada epoca
        if m and m.group(1) != m.group(2):
            continue
        print(linea, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Fallo (codigo {p.returncode}): {' '.join(cmd[:4])} ...")


def entorno_py311():
    env = dict(os.environ)
    env.pop("PYTHONPATH", None)  # que no se mezclen paquetes del Python 3.13 de Colab
    env["MPLBACKEND"] = "Agg"
    env["PYTHONUNBUFFERED"] = "1"
    env["TF_CPP_MIN_LOG_LEVEL"] = "1"
    nvidia = f"{VENV}/lib/python3.11/site-packages/nvidia"
    env["LD_LIBRARY_PATH"] = ":".join(glob.glob(f"{nvidia}/*/lib") + [env.get("LD_LIBRARY_PATH", "")])
    env["PATH"] = f"{nvidia}/cuda_nvcc/bin:" + env.get("PATH", "")
    return env


@register_cell_magic
def py311(linea, celda):
    """Ejecuta la celda con el Python 3.11 de /content/py311."""
    with open("_celda_py311.py", "w", encoding="utf-8") as f:
        f.write(celda)
    correr([PY311, "-u", "_celda_py311.py"], env=entorno_py311())


correr([sys.executable, "-m", "pip", "install", "-q", "roboflow", "uv"])
if not os.path.exists(PY311):
    correr([sys.executable, "-m", "uv", "venv", "--python", "3.11", VENV])
correr([sys.executable, "-m", "uv", "pip", "install", "--python", PY311, *DEPENDENCIAS_PY311])
correr([PY311, "-c",
        "import tensorflow as tf, mediapipe_model_maker; g = tf.config.list_physical_devices('GPU'); "
        "print('Python 3.11 listo. TensorFlow', tf.__version__, 'GPU:', g); "
        "assert g, 'TensorFlow 2.15 no ve la GPU: el entrenamiento seria demasiado lento.'"],
       env=entorno_py311())

if not os.path.exists("unificar_etiquetas.py"):
    r = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", RAMA_REPO, "https://github.com/Ezy01v1/EzyBall", "EzyBall_repo"],
        capture_output=True, text=True,
    )
    origen = os.path.join("EzyBall_repo", "training", "unificar_etiquetas.py")
    if r.returncode == 0 and os.path.exists(origen):
        shutil.copy(origen, "unificar_etiquetas.py")
    else:
        print("No se pudo clonar el repo (puede ser privado). Sube el archivo training/unificar_etiquetas.py:")
        from google.colab import files
        subidos = files.upload()
        nombre = next((n for n in subidos if n.endswith("unificar_etiquetas.py")), None)
        if nombre is None:
            raise SystemExit("No se subio unificar_etiquetas.py; no se puede continuar.")
        if nombre != "unificar_etiquetas.py":
            os.replace(nombre, "unificar_etiquetas.py")

sys.path.insert(0, os.getcwd())
from unificar_etiquetas import unificar_coco, resumen, ErrorDataset
print("Entorno listo.")

## 3. Clave de Roboflow
Se lee del secreto `ROBOFLOW_API_KEY` de Colab; si no existe, se pide por teclado.

In [ ]:
try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    import getpass
    ROBOFLOW_API_KEY = getpass.getpass("Clave de API de Roboflow: ")
assert ROBOFLOW_API_KEY, "Falta la clave de Roboflow."

## 4. Descarga
Descarga cada dataset en formato COCO.

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
descargados = []
for d in DATASETS:
    try:
        proyecto = rf.workspace(d["workspace"]).project(d["project"])
        ds = proyecto.version(d["version"]).download("coco")
    except Exception as e:
        raise SystemExit(f"No se pudo descargar {d}: {e}")
    descargados.append((f"{d['project']}_v{d['version']}", ds.location))
    print("Descargado:", d, "->", ds.location)

## 5. Unificacion
Convierte las etiquetas a `aro`/`balon`, fusiona los datasets (reindexando ids y prefijando nombres de archivo) y deja `datos/<split>/images/` + `datos/<split>/labels.json`, que es lo que pide Model Maker.

In [ ]:
import json, shutil
from pathlib import Path

SPLITS = ["train", "valid", "test"]
shutil.rmtree("datos", ignore_errors=True)

fusion = {}
for split in SPLITS:
    imagenes, anotaciones, categorias = [], [], None
    copias = []
    for prefijo, ruta in descargados:
        carpeta = Path(ruta) / split
        ann_path = carpeta / "_annotations.coco.json"
        if not ann_path.exists():
            print(f"[{prefijo}] sin split '{split}', se omite")
            continue
        coco = json.load(open(ann_path, encoding="utf-8"))
        try:
            coco = unificar_coco(coco)
        except ErrorDataset as e:
            raise SystemExit(f"[{prefijo}/{split}] {e}")
        mapa_id = {}
        for img in coco["images"]:
            nuevo_id = len(imagenes) + 1
            mapa_id[img["id"]] = nuevo_id
            nuevo_nombre = f"{prefijo}__{img['file_name']}"
            copias.append((carpeta / img["file_name"], nuevo_nombre))
            imagenes.append({**img, "id": nuevo_id, "file_name": nuevo_nombre})
        for ann in coco["annotations"]:
            anotaciones.append({**ann, "id": len(anotaciones) + 1, "image_id": mapa_id[ann["image_id"]]})
        categorias = coco["categories"]
    if not imagenes:
        fusion[split] = None
        continue
    destino = Path("datos") / split
    (destino / "images").mkdir(parents=True)
    for origen, nombre in copias:
        shutil.copy(origen, destino / "images" / nombre)
    salida = {"images": imagenes, "categories": categorias, "annotations": anotaciones}
    json.dump(salida, open(destino / "labels.json", "w", encoding="utf-8"), ensure_ascii=False)
    fusion[split] = salida
    print(split, resumen(salida))

assert fusion["train"] and fusion["valid"], "Hacen falta los splits train y valid."
if fusion["test"] is None:
    print("Aviso: no hay split test; se usara valid para evaluar.")

# Configuracion para las celdas %%py311 (otro proceso: no ven las variables de este)
test_dir = "datos/test" if fusion["test"] else "datos/valid"
RUTA_TFLITE = os.path.join("export", "hoop-ball.tflite")
json.dump(
    {"epocas": EPOCAS, "batch": BATCH, "lr": LR, "max_mb": MAX_MB, "test_dir": test_dir, "ruta_tflite": RUTA_TFLITE},
    open("config.json", "w", encoding="utf-8"),
)
print("Configuracion guardada en config.json")

## 6-8. Entrenamiento, evaluacion y exportacion int8
Se ejecuta con el Python 3.11 (`%%py311`) en un solo proceso. Entrena EfficientDet-Lite0 con los hiperparametros de la celda 1, imprime el mAP general y el AP por clase sobre el split de test, exporta con cuantizacion int8 y comprueba que pesa menos de `MAX_MB`. Es la celda que mas tarda.

In [ ]:
%%py311
import json, os, shutil
from mediapipe_model_maker import object_detector, quantization

cfg = json.load(open("config.json", encoding="utf-8"))
BATCH = cfg["batch"]

# Evita reutilizar TFRecords viejos si cambian los DATASETS entre ejecuciones
shutil.rmtree("cache", ignore_errors=True)
shutil.rmtree("export", ignore_errors=True)

train_data = object_detector.Dataset.from_coco_folder("datos/train", cache_dir="cache/train")
validation_data = object_detector.Dataset.from_coco_folder("datos/valid", cache_dir="cache/valid")
test_data = object_detector.Dataset.from_coco_folder(cfg["test_dir"], cache_dir="cache/test")

options = object_detector.ObjectDetectorOptions(
    supported_model=object_detector.SupportedModels.EFFICIENTDET_LITE0,
    hparams=object_detector.HParams(
        epochs=cfg["epocas"], batch_size=BATCH, learning_rate=cfg["lr"], export_dir="export"
    ),
)
model = object_detector.ObjectDetector.create(
    train_data=train_data, validation_data=validation_data, options=options
)

# Evaluacion
loss, metricas = model.evaluate(test_data, batch_size=BATCH)
print(f"Loss: {loss}")
print("Todas las metricas COCO:")
for k, v in metricas.items():
    print(f"  {k}: {v}")
print()
print("mAP general (AP):", metricas.get("AP"))
for k, v in metricas.items():
    if k.startswith("AP_/") or k.lower() in ("ap_aro", "ap_balon"):
        print("AP por clase", k, ":", v)

# Exportacion int8
model.export_model(
    "hoop-ball.tflite",
    quantization_config=quantization.QuantizationConfig.for_int8(representative_data=train_data),
)
RUTA_TFLITE = cfg["ruta_tflite"]
assert os.path.exists(RUTA_TFLITE), f"No se genero {RUTA_TFLITE}; revisa la salida de la celda."
mb = os.path.getsize(RUTA_TFLITE) / 1e6
print(f"Tamano: {mb:.2f} MB")
assert mb < cfg["max_mb"], f"El modelo pesa {mb:.2f} MB y el maximo es {cfg['max_mb']} MB"

## 9. Inspeccion
**Pega la salida de esta celda en la conversacion**: sirve para rellenar el perfil del modelo en la app. Tambien genera `labels.txt` y ejecuta una inferencia de prueba (se ejecuta con el Python 3.11).

In [ ]:
%%py311
import json, os
cfg = json.load(open("config.json", encoding="utf-8"))
RUTA_TFLITE = cfg["ruta_tflite"]
test_dir = cfg["test_dir"]

import zipfile
import numpy as np
import tensorflow as tf

interp = tf.lite.Interpreter(model_path=RUTA_TFLITE)
interp.allocate_tensors()
entrada = interp.get_input_details()[0]
salidas = interp.get_output_details()

print("===== INSPECCION DEL MODELO =====")
print("Tipo de entrada:", np.dtype(entrada["dtype"]).name)
print("Forma de entrada:", list(entrada["shape"]))
print("Cuantizacion de entrada (escala, punto cero):", entrada["quantization"])
print("Salidas:")
for i, s in enumerate(salidas):
    print(f"  indice {i}: nombre={s['name']} forma={list(s['shape'])} tipo={np.dtype(s['dtype']).name} cuantizacion={s['quantization']}")

# Etiquetas: Model Maker las incrusta en el .tflite (que es tambien un zip)
etiquetas = None
try:
    with zipfile.ZipFile(RUTA_TFLITE) as z:
        for n in z.namelist():
            if n.endswith(".txt"):
                etiquetas = z.read(n).decode("utf-8").splitlines()
                break
except zipfile.BadZipFile:
    pass
if not etiquetas:
    print("AVISO: no se encontraron etiquetas en el modelo; se usa el orden por defecto.")
    etiquetas = ["aro", "balon"]
with open("labels.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(etiquetas) + "\n")
print("Etiquetas incrustadas, tal como estan guardadas (indice: texto):")
for i, e in enumerate(etiquetas):
    print(f"  {i}: {e!r}")

# Deteccion del formato de salida
formas = [list(s["shape"]) for s in salidas]
postprocesado = len(salidas) == 4 and any(f == [1] for f in formas)
if postprocesado:
    print("FORMATO: postprocesado (4 salidas: cajas, clases, scores, conteo)")
else:
    print("FORMATO: crudo (anclas, requiere decodificar + NMS)")
    try:
        from mediapipe.tasks.python.metadata import metadata as md
        meta = md.MetadataDisplayer.with_model_file(RUTA_TFLITE).get_metadata_json()
        print("Metadatos incrustados (buscar anchors / decodificacion / escalas):")
        print(meta)
    except Exception as e:
        print("No se pudieron leer los metadatos incrustados:", e)

# Ids de clase CRUDOS que emite el modelo (pueden tener desplazamiento por fondo)
print("--- IDS DE CLASE CRUDOS ---")
try:
    if not postprocesado:
        print("Esta comprobacion no aplica: el formato es crudo (anclas), no hay ids de clase ya decodificados en las salidas.")
    else:
        import glob as _glob
        from PIL import Image as _Image
        _rutas = sorted(_glob.glob(os.path.join(test_dir, "images", "*")))
        if not _rutas:
            print("No hay imagenes de test para esta comprobacion.")
        else:
            _alto, _ancho = int(entrada["shape"][1]), int(entrada["shape"][2])
            _x = np.asarray(_Image.open(_rutas[0]).convert("RGB").resize((_ancho, _alto)), dtype=np.float32)
            _dt = np.dtype(entrada["dtype"])
            if _dt.kind in "iu":
                _esc, _cero = entrada["quantization"]
                if not _esc:
                    _esc, _cero = 1.0, 0
                _info = np.iinfo(_dt)
                _x = np.clip(np.round(_x / _esc + _cero), _info.min, _info.max).astype(_dt)
            else:
                _x = _x.astype(_dt)
            interp.set_tensor(entrada["index"], _x[None, ...])
            interp.invoke()
            _out = [interp.get_tensor(s["index"]) for s in salidas]
            _cajas = _clases = _scores = None
            for _o in _out:
                if _o.ndim == 3 and _o.shape[-1] == 4:
                    _cajas = _o[0]
                elif _o.ndim == 2 and _o.size > 1 and _cajas is not None:
                    pass
            _2d = [_o[0] for _o, s in zip(_out, salidas) if _o.ndim == 2]
            # Entre las dos salidas 2D, los scores estan en [0,1] y las clases son enteros
            for _v in _2d:
                if np.all(_v >= 0) and np.all(_v <= 1.0001) and not np.allclose(_v, np.round(_v)):
                    _scores = _v
                elif _clases is None:
                    _clases = _v
            if _cajas is None or _clases is None or _scores is None:
                print("No se pudieron identificar cajas/clases/scores en las salidas:", [list(o.shape) for o in _out])
            else:
                print("Imagen de prueba:", _rutas[0])
                print("Detecciones crudas del .tflite (top 5):")
                for k in np.argsort(-_scores)[:5]:
                    print(f"  clase_cruda={int(round(float(_clases[k])))} score={float(_scores[k]):.3f} caja={[round(float(c), 3) for c in _cajas[k]]}")
                try:
                    import mediapipe as _mp
                    from mediapipe.tasks import python as _mpp
                    from mediapipe.tasks.python import vision as _vis
                    _det = _vis.ObjectDetector.create_from_options(_vis.ObjectDetectorOptions(
                        base_options=_mpp.BaseOptions(model_asset_path=RUTA_TFLITE),
                        score_threshold=0.3, max_results=5))
                    _r = _det.detect(_mp.Image.create_from_file(_rutas[0]))
                    print("MediaPipe ObjectDetector sobre la misma imagen:")
                    for _d in _r.detections:
                        _c = _d.categories[0]
                        print(f"  category_name={_c.category_name} score={_c.score:.3f}")
                    if not _r.detections:
                        print("  (sin detecciones sobre 0.3)")
                except Exception as e:
                    print("No se pudo ejecutar MediaPipe para comparar:", e)
                print(">>> IMPORTANTE: los valores de clase_cruda de arriba son los que van en claseAro / claseBalon en la app. "
                      "Compara cada clase_cruda con el category_name de MediaPipe para saber cual es aro y cual balon "
                      "(0-based: aro=0, balon=1; con desplazamiento de fondo: aro=1, balon=2).")
except Exception as e:
    print("No se pudieron obtener los ids de clase crudos:", repr(e))
print("===== FIN =====")

# Comprobacion visual con mediapipe (decodifica ambos formatos)
import glob
from PIL import Image, ImageDraw

rutas = sorted(glob.glob(os.path.join(test_dir, "images", "*")))
if rutas:
    try:
        import mediapipe as mp
        from mediapipe.tasks import python as mp_python
        from mediapipe.tasks.python import vision
        opciones = vision.ObjectDetectorOptions(
            base_options=mp_python.BaseOptions(model_asset_path=RUTA_TFLITE),
            score_threshold=0.3, max_results=10,
        )
        detector = vision.ObjectDetector.create_from_options(opciones)
        resultado = detector.detect(mp.Image.create_from_file(rutas[0]))
        img = Image.open(rutas[0]).convert("RGB")
        dib = ImageDraw.Draw(img)
        for d in resultado.detections:
            bb = d.bounding_box
            cat = d.categories[0]
            dib.rectangle([bb.origin_x, bb.origin_y, bb.origin_x + bb.width, bb.origin_y + bb.height], outline="red", width=3)
            dib.text((bb.origin_x + 4, bb.origin_y + 4), f"{cat.category_name}:{cat.score:.2f}", fill="red")
            print("Deteccion:", cat.category_name, round(cat.score, 3), (bb.origin_x, bb.origin_y, bb.width, bb.height))
        if not resultado.detections:
            print("Sin detecciones sobre el umbral 0.3 en esta imagen.")
        img.save("prueba_deteccion.png")
        print("Imagen de prueba guardada en prueba_deteccion.png")
    except Exception as e:
        print("No se pudo ejecutar la comprobacion visual:", e)
else:
    print("No hay imagenes de test para la comprobacion visual.")

In [ ]:
from IPython.display import Image as _Img, display
if os.path.exists("prueba_deteccion.png"):
    display(_Img("prueba_deteccion.png"))
else:
    print("No hay imagen de prueba (revisa la salida de la celda 9).")

## 10. Descarga
Descarga `hoop-ball.tflite` y `labels.txt`. Copia el `.tflite` a `assets/models/` en la app.

In [ ]:
from google.colab import files
files.download(RUTA_TFLITE)
files.download("labels.txt")